# Practical Exercise 4: Knowledge Engineering
## A Knowledge-Based Autonomous Trailer Agent

Practical Exercise 3 found the shortest Nairobi–Kisumu route with Uniform Cost Search (UCS) on a fixed graph. Here the trailer becomes a **knowledge-based agent**: it stores facts and rules in a knowledge base (KB), answers queries from it, and plans routes using only the roads the KB says are usable.

| | Route finder (Practical Exercise 3) | Knowledge-based agent (Practical Exercise 4) |
|---|---|---|
| World model | Fixed graph in code | Facts in a KB |
| Road conditions | Not represented | Facts + rules |
| Change in the world | Edit the code | Tell the KB, replan |

The notebook follows the lecturer's ten knowledge engineering steps and techniques (predicates as dict keys, a rules dict with a cautious default, test-case loops). The section after Step 9 is an **optional extension**.

### Setup
`heapq` (priority queue) and `count` (tie-breaker) are used by UCS, as in Practical Exercise 3.

In [1]:
# heapq gives us a priority queue: UCS uses it to pick the cheapest waiting route.
import heapq

# count() produces 0, 1, 2, ... and is used to break ties between equal-cost queue entries.
from itertools import count

## Step 1: Identify the Task

Deliver cargo from the trailer's current town to its destination along the shortest **usable** route.

| | |
|---|---|
| **Environment** | 10 towns, 12 two-way roads (Practical Exercise 3); road conditions can change |
| **Agent** | The autonomous trailer |
| **Information** | Towns, roads, distances, road conditions, location, destination |
| **Objective** | Reach the destination on usable roads with minimum distance |
| **Decisions** | Is a road usable? What is reachable? Which route? Replan after a change? |

The best route depends on what the agent currently knows, so knowledge is kept separate from reasoning. **Assumptions:** town-level, two-way roads with equal distances, conditions treated as certain.

## Step 2: Assemble Relevant Knowledge

- **Facts (environment, can change):** towns, roads and distances, road conditions (initially all Open), current location, destination.
- **Rules (general):** usable conditions may be travelled and unusable ones may not; unknown conditions are unusable; a town is reachable through a chain of usable roads; take the lowest-cost route.

No APIs or web scraping were used. Road data is copied unchanged from Practical Exercise 3.

In [2]:
# Environment knowledge carried over unchanged from Practical Exercise 3.
# Each tuple reads: (first town, second town, road distance in km).
roadConnections = [
    ("Nairobi", "Nakuru", 161),
    ("Nairobi", "Nyeri", 151),
    ("Nyeri", "Nyahururu", 104),
    ("Nyahururu", "Nakuru", 64),
    ("Nakuru", "Mau Summit", 62),
    ("Mau Summit", "Kericho", 50),
    ("Kericho", "Kisumu", 84),
    ("Nairobi", "Narok", 141),
    ("Narok", "Bomet", 81),
    ("Bomet", "Kericho", 77),
    ("Bomet", "Kisii", 83),
    ("Kisii", "Kisumu", 114),
]

print(f"{'Starting Town':<15}{'Destination':<15}{'Distance':>10}")
print("-" * 40)
for firstTown, secondTown, roadDistance in roadConnections:
    print(f"{firstTown:<15}{secondTown:<15}{roadDistance:>7} km")

townsMentioned = set()
for firstTown, secondTown, roadDistance in roadConnections:
    townsMentioned.add(firstTown)
    townsMentioned.add(secondTown)

print(f"\n{len(townsMentioned)} towns and {len(roadConnections)} road connections.")

Starting Town  Destination      Distance
----------------------------------------
Nairobi        Nakuru             161 km
Nairobi        Nyeri              151 km
Nyeri          Nyahururu          104 km
Nyahururu      Nakuru              64 km
Nakuru         Mau Summit          62 km
Mau Summit     Kericho             50 km
Kericho        Kisumu              84 km
Nairobi        Narok              141 km
Narok          Bomet               81 km
Bomet          Kericho             77 km
Bomet          Kisii               83 km
Kisii          Kisumu             114 km

10 towns and 12 road connections.


## Step 3: Define the Vocabulary

| Predicate | Meaning | Stored as |
|---|---|---|
| `Town(x)` | x is a town | `knowledgeBase["Town"]` (set) |
| `Road(x, y, d)` | road from x to y, d km | `knowledgeBase["Road"][(x, y)] = d` |
| `RoadCondition(x, y, c)` | road x→y is in condition c | `knowledgeBase["RoadCondition"][(x, y)] = c` |
| `At(Trailer, x)` | trailer is in x | `knowledgeBase["At"]["Trailer"] = x` |
| `Destination(Trailer, x)` | cargo goes to x | `knowledgeBase["Destination"]["Trailer"] = x` |

**Derived by rules (not stored):** `CanUse(x, y)` = `RoadOpen(x, y)`, `RoadClosed(x, y)`, `Reachable(x, y)`.

**Constants:** `Trailer`; the ten towns; the conditions Open, Wet, Under Repair, Closed, Flooded, Weight Restricted. Each two-way road is stored as two directed facts.

In [3]:
# The knowledge base. Each key is the name of a predicate and each value holds
# the facts we currently know for that predicate.
knowledgeBase = {
    "Town": set(),         # Town(x)
    "Road": {},            # Road(x, y, d): (x, y) -> d
    "RoadCondition": {},   # RoadCondition(x, y, c): (x, y) -> c
    "At": {},              # At(Trailer, x): "Trailer" -> x
    "Destination": {},     # Destination(Trailer, x): "Trailer" -> x
}

# The constant that names our agent.
trailer = "Trailer"

`normalizeName()` keeps constants consistent (`" mau summit "` → `Mau Summit`), like `.capitalize()` in the reference.

In [4]:
def normalizeName(name):
    """Write a constant in the standard form used by the KB, e.g. '  mau summit ' -> 'Mau Summit'."""
    return name.strip().title()


print(normalizeName("nakuru"))
print(normalizeName("  mau summit "))
print(normalizeName("WEIGHT RESTRICTED"))

Nakuru
Mau Summit
Weight Restricted


## Step 4: Encode General Knowledge about the Domain

### 4.1 Road condition rules
Condition → usable or not, plus advice. This is the counterpart of the lecturer's `rules` dict. Unknown conditions fall back to a cautious "unusable" default. `Weight Restricted` covers trailer suitability.

In [5]:
# General rules about road conditions: condition -> whether the trailer may use the road, and advice.
roadConditionRules = {
    "Open":              {"usable": True,  "advice": "Usable: the road is open to the trailer."},
    "Wet":               {"usable": True,  "advice": "Usable: reduce speed, the surface is slippery."},
    "Under Repair":      {"usable": True,  "advice": "Usable: expect delays from roadworks."},
    "Closed":            {"usable": False, "advice": "Not usable: the road is closed to all traffic."},
    "Flooded":           {"usable": False, "advice": "Not usable: the trailer cannot safely cross flood water."},
    "Weight Restricted": {"usable": False, "advice": "Not usable: the loaded trailer exceeds the weight limit."},
}

# Cautious default for any condition we have no rule for.
unknownConditionRule = {"usable": False, "advice": "Unknown condition: treat the road as unusable until confirmed safe."}


def getConditionRule(condition):
    """Look up the rule for a road condition, falling back to the cautious default (like rules.get in the reference)."""
    return roadConditionRules.get(normalizeName(condition), unknownConditionRule)

### 4.2 Rules for using roads
- `Road(x, y, d)` ∧ `RoadCondition(x, y, c)` ∧ usable(c) ⇒ `CanUse(x, y)` (= `RoadOpen`)
- `Road(x, y, d)` ∧ ¬`CanUse(x, y)` ⇒ `RoadClosed(x, y)`

In [6]:
def roadExists(fromTown, toTown):
    """True if the fact Road(fromTown, toTown, d) is in the KB for some distance d."""
    return (normalizeName(fromTown), normalizeName(toTown)) in knowledgeBase["Road"]


def canUseRoad(fromTown, toTown):
    """
    CanUse rule: the road must exist AND the rule for its current condition must say it is usable.
    """
    fromTown = normalizeName(fromTown)
    toTown = normalizeName(toTown)

    if not roadExists(fromTown, toTown):
        return False

    condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
    return getConditionRule(condition)["usable"]


def isRoadOpen(fromTown, toTown):
    """RoadOpen(x, y) means the same as CanUse(x, y)."""
    return canUseRoad(fromTown, toTown)


def isRoadClosed(fromTown, toTown):
    """RoadClosed rule: a known road that the trailer is not allowed to use."""
    return roadExists(fromTown, toTown) and not canUseRoad(fromTown, toTown)

### 4.3 Rules for reachability
1. `CanUse(x, y)` ⇒ `Reachable(x, y)`
2. `Reachable(x, y)` ∧ `CanUse(y, z)` ⇒ `Reachable(x, z)`

Applied by forward chaining until no new town is inferred.

In [7]:
def getUsableNeighbours(town):
    """Return (neighbour, distance) for every road out of town that the CanUse rule allows."""
    town = normalizeName(town)
    usableNeighbours = []

    for (fromTown, toTown), roadDistance in knowledgeBase["Road"].items():
        if fromTown == town and canUseRoad(fromTown, toTown):
            usableNeighbours.append((toTown, roadDistance))

    # Same ordering as Practical Exercise 3's buildGraph(): shortest road first, then by name.
    usableNeighbours.sort(key=lambda connection: (connection[1], connection[0]))
    return usableNeighbours


def findReachableTowns(startTown):
    """
    Forward chaining over the two Reachable rules.
    townsToExpand holds towns we have inferred as reachable but whose roads we have not yet checked.
    """
    startTown = normalizeName(startTown)
    reachableTowns = set()
    townsToExpand = [startTown]

    while len(townsToExpand) > 0:
        town = townsToExpand.pop()
        for neighbourTown, roadDistance in getUsableNeighbours(town):
            # Only record a town the first time we infer it; this also stops us going round cycles.
            if neighbourTown != startTown and neighbourTown not in reachableTowns:
                reachableTowns.add(neighbourTown)
                townsToExpand.append(neighbourTown)

    return reachableTowns

### 4.4 Rule for choosing a route
Take the lowest-cost usable route. UCS carries this out in Step 8.

## Step 5: Add New Facts to the Knowledge Base
`addTown`, `addRoad` (both directions, initially Open), `setRoadCondition` (existing roads only) and `showKnowledgeBase`. Compare the lecturer's `add_new_fact` and `show_kb`.

In [8]:
def addTown(town):
    """Tell the KB the fact Town(town). Returns the normalised name."""
    town = normalizeName(town)
    knowledgeBase["Town"].add(town)
    return town


def addRoad(firstTown, secondTown, roadDistance, condition="Open"):
    """Tell the KB about a two-way road: Road(x, y, d), Road(y, x, d) and their conditions."""
    firstTown = addTown(firstTown)
    secondTown = addTown(secondTown)
    condition = normalizeName(condition)

    for fromTown, toTown in [(firstTown, secondTown), (secondTown, firstTown)]:
        knowledgeBase["Road"][(fromTown, toTown)] = roadDistance
        knowledgeBase["RoadCondition"][(fromTown, toTown)] = condition


def setRoadCondition(firstTown, secondTown, condition):
    """Update RoadCondition for both directions of an existing road. Returns True if the KB changed."""
    firstTown = normalizeName(firstTown)
    secondTown = normalizeName(secondTown)
    condition = normalizeName(condition)

    if not roadExists(firstTown, secondTown):
        print(f"  Rejected: the KB has no road between {firstTown} and {secondTown}.")
        return False

    knowledgeBase["RoadCondition"][(firstTown, secondTown)] = condition
    knowledgeBase["RoadCondition"][(secondTown, firstTown)] = condition
    return True

In [9]:
def showKnowledgeBase():
    """Print every fact currently in the KB, grouped by predicate (compare the lecturer's show_kb)."""
    print("\n--- Current Knowledge Base ---")
    print(f"Town facts ({len(knowledgeBase['Town'])}): {', '.join(sorted(knowledgeBase['Town']))}")

    print(f"\nRoad facts ({len(knowledgeBase['Road'])} directed; each two-way road is shown once):")
    for (fromTown, toTown), roadDistance in sorted(knowledgeBase["Road"].items()):
        if fromTown < toTown:
            condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
            roadFact = f"Road({fromTown}, {toTown}, {roadDistance})"
            print(f"  {roadFact:<34} RoadCondition: {condition}")

    print("\nAt facts:")
    for agentName, town in knowledgeBase["At"].items():
        print(f"  At({agentName}, {town})")
    if len(knowledgeBase["At"]) == 0:
        print("  (none yet)")

    print("Destination facts:")
    for agentName, town in knowledgeBase["Destination"].items():
        print(f"  Destination({agentName}, {town})")
    if len(knowledgeBase["Destination"]) == 0:
        print("  (none yet)")
    print("------------------------------")

Interactive version (as in the reference). The call is commented out so the notebook runs unattended.

In [10]:
def addRoadFromUser():
    """Interactive version of addRoad, like the lecturer's add_fact()."""
    firstTown = input("First town: ")
    secondTown = input("Second town: ")
    roadDistance = int(input("Distance in km: "))
    addRoad(firstTown, secondTown, roadDistance)
    print(f"New fact added: Road({normalizeName(firstTown)}, {normalizeName(secondTown)}, {roadDistance}) in both directions")


# addRoadFromUser()   # uncomment to add a road interactively

### 5.1 Populate the KB with the Practical Exercise 3 road network
Expect 10 towns and 24 directed Road facts.

In [11]:
for firstTown, secondTown, roadDistance in roadConnections:
    addRoad(firstTown, secondTown, roadDistance)

showKnowledgeBase()

assert len(knowledgeBase["Town"]) == 10, "Expected 10 towns from Practical Exercise 3."
assert len(knowledgeBase["Road"]) == 24, "Expected 12 two-way roads = 24 directed Road facts."
assert len(knowledgeBase["RoadCondition"]) == 24, "Every Road fact should have a RoadCondition fact."
print("\nCheck passed: the KB holds the 10 towns and 12 two-way roads from Practical Exercise 3.")


--- Current Knowledge Base ---
Town facts (10): Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nairobi, Nakuru, Narok, Nyahururu, Nyeri

Road facts (24 directed; each two-way road is shown once):
  Road(Bomet, Kericho, 77)           RoadCondition: Open
  Road(Bomet, Kisii, 83)             RoadCondition: Open
  Road(Bomet, Narok, 81)             RoadCondition: Open
  Road(Kericho, Kisumu, 84)          RoadCondition: Open
  Road(Kericho, Mau Summit, 50)      RoadCondition: Open
  Road(Kisii, Kisumu, 114)           RoadCondition: Open
  Road(Mau Summit, Nakuru, 62)       RoadCondition: Open
  Road(Nairobi, Nakuru, 161)         RoadCondition: Open
  Road(Nairobi, Narok, 141)          RoadCondition: Open
  Road(Nairobi, Nyeri, 151)          RoadCondition: Open
  Road(Nakuru, Nyahururu, 64)        RoadCondition: Open
  Road(Nyahururu, Nyeri, 104)        RoadCondition: Open

At facts:
  (none yet)
Destination facts:
  (none yet)
------------------------------

Check passed: the KB holds the 10 t

### 5.2 Retrieving and updating facts
The Narok–Bomet road is set to Wet, then back to Open. Usability is inferred each time, not stored.

In [12]:
# Retrieving facts
print("Road(Nairobi, Nakuru, d): d =", knowledgeBase["Road"][("Nairobi", "Nakuru")], "km")
print("Road(Nakuru, Nairobi, d): d =", knowledgeBase["Road"][("Nakuru", "Nairobi")], "km  (the reverse direction)")
print("RoadCondition(Nairobi, Nakuru):", knowledgeBase["RoadCondition"][("Nairobi", "Nakuru")])
print("Town(Bomet)?", "Bomet" in knowledgeBase["Town"])

# Updating a fact: rain is reported on Narok-Bomet.
print("\nUpdate: the Narok-Bomet road is reported wet.")
setRoadCondition("Narok", "Bomet", "Wet")
print("  RoadCondition(Narok, Bomet):", knowledgeBase["RoadCondition"][("Narok", "Bomet")])
print("  Rule applied:", getConditionRule("Wet")["advice"])
print("  CanUse(Narok, Bomet)?", canUseRoad("Narok", "Bomet"))

# The rain clears, so the original fact is restored.
print("\nUpdate: the Narok-Bomet road has dried.")
setRoadCondition("Narok", "Bomet", "Open")
print("  RoadCondition(Narok, Bomet):", knowledgeBase["RoadCondition"][("Narok", "Bomet")])

Road(Nairobi, Nakuru, d): d = 161 km
Road(Nakuru, Nairobi, d): d = 161 km  (the reverse direction)
RoadCondition(Nairobi, Nakuru): Open
Town(Bomet)? True

Update: the Narok-Bomet road is reported wet.
  RoadCondition(Narok, Bomet): Wet
  Rule applied: Usable: reduce speed, the surface is slippery.
  CanUse(Narok, Bomet)? True

Update: the Narok-Bomet road has dried.
  RoadCondition(Narok, Bomet): Open


## Step 6: Encode the Specific Problem Instance
`At(Trailer, Nairobi)` and `Destination(Trailer, Kisumu)`. Only known towns are accepted.

In [13]:
def setTrailerLocation(town):
    """Tell the KB At(Trailer, town). Only known towns are accepted."""
    town = normalizeName(town)
    if town not in knowledgeBase["Town"]:
        print(f"  Rejected: '{town}' is not a known town, so At({trailer}, {town}) was not added.")
        return False
    knowledgeBase["At"][trailer] = town
    return True


def setDestination(town):
    """Tell the KB Destination(Trailer, town). Only known towns are accepted."""
    town = normalizeName(town)
    if town not in knowledgeBase["Town"]:
        print(f"  Rejected: '{town}' is not a known town, so Destination({trailer}, {town}) was not added.")
        return False
    knowledgeBase["Destination"][trailer] = town
    return True

In [14]:
# The specific problem instance (same as Practical Exercise 3's initialState and goalState).
# startingLocation = input("Where is the trailer now? ")   # interactive alternative
startingLocation = "Nairobi"
deliveryDestination = "Kisumu"

setTrailerLocation(startingLocation)
setDestination(deliveryDestination)

showKnowledgeBase()


--- Current Knowledge Base ---
Town facts (10): Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nairobi, Nakuru, Narok, Nyahururu, Nyeri

Road facts (24 directed; each two-way road is shown once):
  Road(Bomet, Kericho, 77)           RoadCondition: Open
  Road(Bomet, Kisii, 83)             RoadCondition: Open
  Road(Bomet, Narok, 81)             RoadCondition: Open
  Road(Kericho, Kisumu, 84)          RoadCondition: Open
  Road(Kericho, Mau Summit, 50)      RoadCondition: Open
  Road(Kisii, Kisumu, 114)           RoadCondition: Open
  Road(Mau Summit, Nakuru, 62)       RoadCondition: Open
  Road(Nairobi, Nakuru, 161)         RoadCondition: Open
  Road(Nairobi, Narok, 141)          RoadCondition: Open
  Road(Nairobi, Nyeri, 151)          RoadCondition: Open
  Road(Nakuru, Nyahururu, 64)        RoadCondition: Open
  Road(Nyahururu, Nyeri, 104)        RoadCondition: Open

At facts:
  At(Trailer, Nairobi)
Destination facts:
  Destination(Trailer, Kisumu)
------------------------------


## Step 7: Pose Queries and Get Answers
Each answer is labelled **fact lookup** or **inference**. Unknown items return "unknown" instead of failing.

In [15]:
def listTowns():
    """Which towns exist? (fact lookup)"""
    return sorted(knowledgeBase["Town"])


def isKnownTown(town):
    """Is Town(town) in the KB? (fact lookup)"""
    return normalizeName(town) in knowledgeBase["Town"]


def getConnectedTowns(town):
    """Which towns does a Road fact join to this town, whatever the road's condition? (fact lookup)"""
    town = normalizeName(town)
    connectedTowns = []
    for (fromTown, toTown), roadDistance in knowledgeBase["Road"].items():
        if fromTown == town:
            connectedTowns.append((toTown, roadDistance))
    connectedTowns.sort(key=lambda connection: (connection[1], connection[0]))
    return connectedTowns


def getRoadDistance(fromTown, toTown):
    """Distance d from Road(x, y, d), or None if no such road is known. (fact lookup)"""
    return knowledgeBase["Road"].get((normalizeName(fromTown), normalizeName(toTown)))


def describeRoadStatus(fromTown, toTown):
    """Is the road open? Answer with its condition and the advice from the matching rule. (inference)"""
    fromTown = normalizeName(fromTown)
    toTown = normalizeName(toTown)
    if not roadExists(fromTown, toTown):
        return f"Unknown road: the KB has no Road fact from {fromTown} to {toTown}."
    condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
    return f"{condition}. {getConditionRule(condition)['advice']}"

In [16]:
def getCurrentLocation():
    """Where is the trailer? Looks up At(Trailer, x)."""
    return knowledgeBase["At"].get(trailer)


def getDestination():
    """Where must the trailer go? Looks up Destination(Trailer, x)."""
    return knowledgeBase["Destination"].get(trailer)


def canReachDestination():
    """Does Reachable(current location, destination) hold? (inference)"""
    currentLocation = getCurrentLocation()
    destination = getDestination()
    if currentLocation is None or destination is None:
        return False
    if currentLocation == destination:
        return True
    return destination in findReachableTowns(currentLocation)


def listUnavailableRoads():
    """Every road for which RoadClosed(x, y) can be inferred, each two-way road listed once. (inference)"""
    unavailableRoads = []
    for fromTown, toTown in knowledgeBase["Road"]:
        if fromTown < toTown and isRoadClosed(fromTown, toTown):
            condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
            unavailableRoads.append((fromTown, toTown, condition))
    return sorted(unavailableRoads)


def printAnswer(question, answer, howAnswered):
    """Print a question, the agent's answer and how the answer was obtained."""
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   [{howAnswered}]\n")

In [17]:
printAnswer("Which towns exist?", ", ".join(listTowns()), "fact lookup: Town(x)")

nairobiConnections = []
for town, roadDistance in getConnectedTowns("Nairobi"):
    nairobiConnections.append(f"{town} ({roadDistance} km)")
printAnswer("Which towns are connected to Nairobi?", ", ".join(nairobiConnections), "fact lookup: Road(Nairobi, y, d)")

printAnswer("What is the distance between Nairobi and Nakuru?", f"{getRoadDistance('Nairobi', 'Nakuru')} km",
            "fact lookup: Road(Nairobi, Nakuru, d)")

printAnswer("Is the road from Nakuru to Mau Summit open?", describeRoadStatus("Nakuru", "Mau Summit"),
            "inference: RoadCondition + condition rule => RoadOpen")

printAnswer("Where is the trailer?", getCurrentLocation(), "fact lookup: At(Trailer, x)")
printAnswer("What is the destination?", getDestination(), "fact lookup: Destination(Trailer, x)")

directMoves = []
for town, roadDistance in getUsableNeighbours(getCurrentLocation()):
    directMoves.append(f"{town} ({roadDistance} km)")
printAnswer("Which towns can the trailer drive to directly from where it is?", ", ".join(directMoves),
            "inference: CanUse(At, y)")

printAnswer("Which towns are reachable from the current location?", ", ".join(sorted(findReachableTowns(getCurrentLocation()))),
            "inference: Reachable rules, forward chaining")

printAnswer("Can the trailer reach Kisumu?", "Yes" if canReachDestination() else "No",
            "inference: Reachable(At, Destination)")

unavailableRoads = listUnavailableRoads()
printAnswer("Which roads are unavailable?", unavailableRoads if len(unavailableRoads) > 0 else "None: every known road is usable.",
            "inference: RoadClosed rule over all Road facts")

Q: Which towns exist?
A: Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nairobi, Nakuru, Narok, Nyahururu, Nyeri
   [fact lookup: Town(x)]

Q: Which towns are connected to Nairobi?
A: Narok (141 km), Nyeri (151 km), Nakuru (161 km)
   [fact lookup: Road(Nairobi, y, d)]

Q: What is the distance between Nairobi and Nakuru?
A: 161 km
   [fact lookup: Road(Nairobi, Nakuru, d)]

Q: Is the road from Nakuru to Mau Summit open?
A: Open. Usable: the road is open to the trailer.
   [inference: RoadCondition + condition rule => RoadOpen]

Q: Where is the trailer?
A: Nairobi
   [fact lookup: At(Trailer, x)]

Q: What is the destination?
A: Kisumu
   [fact lookup: Destination(Trailer, x)]

Q: Which towns can the trailer drive to directly from where it is?
A: Narok (141 km), Nyeri (151 km), Nakuru (161 km)
   [inference: CanUse(At, y)]

Q: Which towns are reachable from the current location?
A: Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nakuru, Narok, Nyahururu, Nyeri
   [inference: Reachable rules, forw

## Step 8: Querying the Knowledge Base to Make Decisions

```
KB facts → CanUse rule → usable road graph ─┐
KB At / Destination ────────────────────────┴→ UCS (Practical Exercise 3) → route, cost, explanation
```

### 8.1 Build the road graph from the KB
`buildAvailableRoadGraph()` adapts Practical Exercise 3's `buildGraph()`. It keeps the same structure but includes only roads the KB marks as usable. With every road open it should equal the Practical Exercise 3 graph.

In [18]:
def buildAvailableRoadGraph():
    """Adapted from Practical Exercise 3's buildGraph(): same structure, but built from usable roads in the KB."""
    availableRoadGraph = {}
    for town in sorted(knowledgeBase["Town"]):
        availableRoadGraph[town] = getUsableNeighbours(town)
    return availableRoadGraph


def buildGraph(roadConnections):
    """Practical Exercise 3's original graph construction, kept unchanged for comparison."""
    roadGraph = {}
    for firstTown, secondTown, roadDistance in roadConnections:
        if firstTown not in roadGraph:
            roadGraph[firstTown] = []
        if secondTown not in roadGraph:
            roadGraph[secondTown] = []
        # Practical Exercise 3 assumes this road can be travelled in both directions.
        roadGraph[firstTown].append((secondTown, roadDistance))
        roadGraph[secondTown].append((firstTown, roadDistance))

    for town in roadGraph:
        roadGraph[town].sort(key=lambda connection: (connection[1], connection[0]))

    return roadGraph


def countArcs(roadGraph):
    """Number of directed arcs in a graph."""
    arcCount = 0
    for town in roadGraph:
        arcCount = arcCount + len(roadGraph[town])
    return arcCount


availableRoadGraph = buildAvailableRoadGraph()
practicalThreeGraph = buildGraph(roadConnections)

print("Road graph built from the KB (usable roads only):")
for town, neighbours in availableRoadGraph.items():
    neighbourDescriptions = []
    for neighbourTown, roadDistance in neighbours:
        neighbourDescriptions.append(f"{neighbourTown} ({roadDistance})")
    print(f"  {town:<11} -> {', '.join(neighbourDescriptions)}")

print(f"\nDirected arcs in the KB graph: {countArcs(availableRoadGraph)}")
print("Identical to the Practical Exercise 3 graph while every road is open?", availableRoadGraph == practicalThreeGraph)
assert availableRoadGraph == practicalThreeGraph

Road graph built from the KB (usable roads only):
  Bomet       -> Kericho (77), Narok (81), Kisii (83)
  Kericho     -> Mau Summit (50), Bomet (77), Kisumu (84)
  Kisii       -> Bomet (83), Kisumu (114)
  Kisumu      -> Kericho (84), Kisii (114)
  Mau Summit  -> Kericho (50), Nakuru (62)
  Nairobi     -> Narok (141), Nyeri (151), Nakuru (161)
  Nakuru      -> Mau Summit (62), Nyahururu (64), Nairobi (161)
  Narok       -> Bomet (81), Nairobi (141)
  Nyahururu   -> Nakuru (64), Nyeri (104)
  Nyeri       -> Nyahururu (104), Nairobi (151)

Directed arcs in the KB graph: 24
Identical to the Practical Exercise 3 graph while every road is open? True


### 8.2 Helper functions reused from Practical Exercise 3

In [19]:
def printBanner(title, separatorCharacter="="):
    """Print a heading that makes the console output easier to scan."""
    print("\n" + separatorCharacter * 78)
    print(title)
    print(separatorCharacter * 78)


def formatFrontier(frontier, bestCosts, settledTowns):
    """Show the current best entries waiting in the priority queue."""
    liveFrontier = []

    for pathCost, insertionOrder, town in frontier:
        # A town may still have an old queue entry after a cheaper route was found.
        # Only show entries that match the best cost we currently know for that town.
        if town not in settledTowns:
            if pathCost == bestCosts[town]:
                liveFrontier.append((pathCost, insertionOrder, town))

    # This order matches the way the queue chooses its next entry.
    liveFrontier.sort()

    if len(liveFrontier) == 0:
        return "(empty)"

    frontierDescriptions = []
    for pathCost, insertionOrder, town in liveFrontier:
        frontierDescriptions.append(f"{town} [cost= {pathCost} km]")

    return ", ".join(frontierDescriptions)


def reconstructPath(predecessors, destinationTown):
    """Follow the saved previous-town links to rebuild the route."""
    route = []
    currentTown = destinationTown

    while currentTown is not None:
        route.append(currentTown)
        currentTown = predecessors[currentTown]
    # We followed the route backwards, so reverse it to get start -> destination.
    route.reverse()
    return route


def getEdgeCost(roadGraph, startingTown, destinationTown):
    """Return the distance for a road between two neighbouring towns."""
    for neighbourTown, roadDistance in roadGraph[startingTown]:
        if neighbourTown == destinationTown:
            return roadDistance

    raise ValueError(f"No road connects {startingTown} and {destinationTown} in this graph.")

### 8.3 Uniform Cost Search (reused from Practical Exercise 3)
Unchanged Practical Exercise 3 logic (priority-queue frontier, `bestCosts`, `predecessors`, stale-entry skipping, goal test on removal, search tree record). The only addition is a `showTrace` switch.

In [20]:
def uniformCostSearch(roadGraph, startTown, destinationTown, showTrace=True):
    """Practical Exercise 3's Uniform Cost Search, with an option to hide the step-by-step trace."""

    def trace(message=""):
        """Print one line of the search trace, but only if showTrace is True."""
        if showTrace:
            print(message)

    # The queue stores (total distance so far, tie-break number, town).
    tieBreaker = count()
    frontier = [(0, next(tieBreaker), startTown)]

    # bestCosts remembers the cheapest route found to each town.
    bestCosts = {startTown: 0}

    # predecessors remembers the town we came from on that cheapest route.
    predecessors = {startTown: None}

    # Once a town is settled, its cheapest distance is final for this graph.
    settledTowns = set()

    # A record of each road option considered, for printing the search tree later.
    searchTreeRecord = []
    selectedTownCount = 0
    expandedTownCount = 0

    if showTrace:
        printBanner("UNIFORM COST SEARCH")
    trace(f"Start: {startTown}")
    trace(f"Goal:  {destinationTown}")
    trace("Starting cost is 0 km because the trailer has not travelled yet.")
    trace(f"Frontier: {formatFrontier(frontier, bestCosts, settledTowns)}")
    trace("Settled towns: (none yet)")
    trace("\nImportant rule: we check for the goal when it is removed from the frontier.")
    trace("Finding the goal as a neighbour is not enough to stop, because a cheaper")
    trace("route to it might still be waiting to be found.")

    while len(frontier) > 0:
        # Remove whichever entry has the smallest total distance.
        pathCost, insertionOrder, currentTown = heapq.heappop(frontier)

        # If this town was already settled, this queue entry can be ignored.
        if currentTown in settledTowns:
            trace(f"\n[SKIP] {currentTown} at {pathCost} km was already settled.")
            continue

        # A better route may have been found after this older entry was added.
        if pathCost != bestCosts[currentTown]:
            trace(f"\n[SKIP] Old entry for {currentTown}: {pathCost} km.")
            trace(f"       The best known cost is now {bestCosts[currentTown]} km, so this entry is out of date.")
            continue

        selectedTownCount += 1
        if showTrace:
            printBanner(f"STEP {selectedTownCount}: TAKE {currentTown.upper()} FROM THE FRONTIER", "-")
        trace(f"We selected {currentTown} because its current cost is the lowest: {pathCost} km.")
        trace(f"Route so far: {' -> '.join(reconstructPath(predecessors, currentTown))}")
        trace(f"Other options waiting: {formatFrontier(frontier, bestCosts, settledTowns)}")

        # No cheaper route to this town can appear later, so we settle it now.
        settledTowns.add(currentTown)

        # Goal test happens only when a town is taken from the priority queue.
        if currentTown == destinationTown:
            trace(f"\nGOAL TEST: Is {currentTown} the destination {destinationTown}? YES.")
            trace(f"We reached the goal with a total path cost of {pathCost} km.")
            trace("Because this was the lowest-cost entry in the frontier, the route is optimal for this graph.")

            finalPath = reconstructPath(predecessors, destinationTown)
            return finalPath, pathCost, searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns

        trace(f"\nGOAL TEST: Is {currentTown} the destination {destinationTown}? No.")
        trace(f"We will now check the roads leaving {currentTown}.")
        expandedTownCount += 1

        for neighbourTown, roadDistance in roadGraph[currentTown]:
            newPathCost = pathCost + roadDistance

            trace(f"\n  Road option: {currentTown} -> {neighbourTown} ({roadDistance} km)")
            trace(f"  Cost if we take this road: {pathCost} + {roadDistance} = {newPathCost} km.")

            if neighbourTown in settledTowns:
                trace(f"  Decision: ignore it. We've already settled {neighbourTown} at its cheapest cost.")
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "already settled; ignored"))

            elif neighbourTown not in bestCosts:
                # First route found to this town: remember it and add it to the frontier.
                bestCosts[neighbourTown] = newPathCost
                predecessors[neighbourTown] = currentTown
                heapq.heappush(frontier, (newPathCost, next(tieBreaker), neighbourTown))
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "new route"))
                trace(f"  Decision: add {neighbourTown} to the frontier at {newPathCost} km. This is our first route to it.")

            elif newPathCost < bestCosts[neighbourTown]:
                # Cheaper route found: update the cost and the previous town.
                previousBestCost = bestCosts[neighbourTown]
                bestCosts[neighbourTown] = newPathCost
                predecessors[neighbourTown] = currentTown
                heapq.heappush(frontier, (newPathCost, next(tieBreaker), neighbourTown))
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, f"improved from {previousBestCost} km"))
                trace(f"  Decision: keep this route. {newPathCost} km is cheaper than the previous best of {previousBestCost} km.")
                trace("  The old queue entry can stay there for now; we'll skip it if it comes up later.")

            elif newPathCost == bestCosts[neighbourTown]:
                # A tie does not improve the best cost, so no new queue entry is needed.
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "same cost; no update"))
                trace(f"  Decision: the cost ties the current best of {bestCosts[neighbourTown]} km, so no update is needed.")

            else:
                # This option costs more than the route we already know.
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "more expensive; discarded"))
                trace(f"  Decision: discard it. We already know a cheaper route costing {bestCosts[neighbourTown]} km.")

        trace(f"\nFrontier after checking {currentTown}: {formatFrontier(frontier, bestCosts, settledTowns)}")
        trace(f"Towns settled so far: {', '.join(sorted(settledTowns))}")

    # The frontier emptied without settling the goal, so no route exists in this graph.
    trace(f"No route from {startTown} to {destinationTown} was found in this graph.")
    return None, float("inf"), searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns

### 8.4 The agent's decision procedure
`planRoute()`: read `At`/`Destination` → stop if already there → check reachability → build the usable graph → run UCS. `explainPlan()` prints the route table, the reason and the next move.

In [21]:
def planRoute(showTrace=False):
    """The agent's decision procedure: ask the KB, check reachability, build the usable graph, run UCS."""
    currentLocation = getCurrentLocation()
    destination = getDestination()

    plan = {
        "start": currentLocation,
        "destination": destination,
        "route": None,
        "cost": None,
        "roadGraph": None,
        "searchTreeRecord": [],
        "reason": "",
    }

    if currentLocation is None or destination is None:
        plan["reason"] = "The KB does not say where the trailer is or where it must go, so no plan can be made."
        return plan

    if currentLocation == destination:
        plan["route"] = [currentLocation]
        plan["cost"] = 0
        plan["reason"] = f"At({trailer}, {currentLocation}) and Destination({trailer}, {destination}) agree: no movement is needed."
        return plan

    if not canReachDestination():
        plan["reason"] = (f"The reachability rules show no chain of usable roads from {currentLocation} to {destination}. "
                          f"Unavailable roads: {listUnavailableRoads()}")
        return plan

    roadGraph = buildAvailableRoadGraph()
    route, cost, searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns = uniformCostSearch(
        roadGraph, currentLocation, destination, showTrace
    )

    usableRoadCount = countArcs(roadGraph) // 2
    totalRoadCount = len(knowledgeBase["Road"]) // 2
    plan["route"] = route
    plan["cost"] = cost
    plan["roadGraph"] = roadGraph
    plan["searchTreeRecord"] = searchTreeRecord
    plan["reason"] = (f"Using only the {usableRoadCount} of {totalRoadCount} roads the KB marks as usable, "
                      f"UCS settled {selectedTownCount} towns and found this to be the lowest-cost route.")
    return plan

In [22]:
def explainPlan(plan):
    """Present the agent's decision: route table (Practical Exercise 3 layout), reason and next move."""
    printBanner(f"AGENT DECISION: {plan['start']} -> {plan['destination']}")

    if plan["route"] is None:
        print("Decision: do not move. No usable route exists.")
        print("Reason:", plan["reason"])
        return

    route = plan["route"]
    print("Route:", " -> ".join(route))

    if len(route) > 1:
        print()
        print(f"{'Road segment':<34}{'Distance (km)':>16}{'Cumulative (km)':>20}")
        print("-" * 70)
        cumulativeDistance = 0
        distanceParts = []
        for pathIndex in range(len(route) - 1):
            startingTown = route[pathIndex]
            nextTown = route[pathIndex + 1]
            roadDistance = getEdgeCost(plan["roadGraph"], startingTown, nextTown)
            cumulativeDistance = cumulativeDistance + roadDistance
            distanceParts.append(str(roadDistance))
            roadSegment = startingTown + " -> " + nextTown
            print(f"{roadSegment:<34}{roadDistance:>16}{cumulativeDistance:>20}")
        print()
        print("Total distance:", " + ".join(distanceParts), "=", cumulativeDistance, "km")
        print("Matches the cost returned by UCS:", cumulativeDistance == plan["cost"])
        print(f"Next move: drive from {route[0]} to {route[1]}.")

    print("Reason:", plan["reason"])

### 8.5 The agent plans its route from Nairobi to Kisumu
Full UCS trace for the first plan.

In [23]:
baselinePlan = planRoute(showTrace=True)


UNIFORM COST SEARCH
Start: Nairobi
Goal:  Kisumu
Starting cost is 0 km because the trailer has not travelled yet.
Frontier: Nairobi [cost= 0 km]
Settled towns: (none yet)

Important rule: we check for the goal when it is removed from the frontier.
Finding the goal as a neighbour is not enough to stop, because a cheaper
route to it might still be waiting to be found.

------------------------------------------------------------------------------
STEP 1: TAKE NAIROBI FROM THE FRONTIER
------------------------------------------------------------------------------
We selected Nairobi because its current cost is the lowest: 0 km.
Route so far: Nairobi
Other options waiting: (empty)

GOAL TEST: Is Nairobi the destination Kisumu? No.
We will now check the roads leaving Nairobi.

  Road option: Nairobi -> Narok (141 km)
  Cost if we take this road: 0 + 141 = 141 km.
  Decision: add Narok to the frontier at 141 km. This is our first route to it.

  Road option: Nairobi -> Nyeri (151 km)
  Cost

In [24]:
explainPlan(baselinePlan)

assert baselinePlan["route"] == ["Nairobi", "Nakuru", "Mau Summit", "Kericho", "Kisumu"]
assert baselinePlan["cost"] == 357
print("\nCheck passed: the agent chose Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km), as in Practical Exercise 3.")


AGENT DECISION: Nairobi -> Kisumu
Route: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Nakuru                              161                 161
Nakuru -> Mau Summit                            62                 223
Mau Summit -> Kericho                           50                 273
Kericho -> Kisumu                               84                 357

Total distance: 161 + 62 + 50 + 84 = 357 km
Matches the cost returned by UCS: True
Next move: drive from Nairobi to Nakuru.
Reason: Using only the 12 of 12 roads the KB marks as usable, UCS settled 10 towns and found this to be the lowest-cost route.

Check passed: the agent chose Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km), as in Practical Exercise 3.


### 8.6 Verifying the decision (from Practical Exercise 3)
No simple route should be cheaper than the UCS result.

In [25]:
def findAllSimplePaths(roadGraph, currentTown, destinationTown, currentPath, currentPathCost, foundRoutes):
    """Collect every route to the destination without visiting a town twice (Practical Exercise 3)."""
    currentPath.append(currentTown)

    if currentTown == destinationTown:
        # Save a copy, because currentPath changes as the recursion unwinds.
        foundRoutes.append((currentPath.copy(), currentPathCost))
    else:
        for neighbourTown, roadDistance in roadGraph[currentTown]:
            if neighbourTown not in currentPath:
                findAllSimplePaths(roadGraph, neighbourTown, destinationTown,
                                   currentPath, currentPathCost + roadDistance, foundRoutes)

    # Remove this town before returning so other routes can use it.
    currentPath.pop()


def listAllSimpleRoutes(plan):
    """Print every simple route in the plan's graph, cheapest first, and return them."""
    allSimpleRoutes = []
    findAllSimplePaths(plan["roadGraph"], plan["start"], plan["destination"], [], 0, allSimpleRoutes)
    allSimpleRoutes.sort(key=lambda routeDetails: routeDetails[1])

    for routePath, routeCost in allSimpleRoutes:
        marker = "  <-- chosen by the agent" if routePath == plan["route"] else ""
        print(f"{routeCost:>4} km  {' -> '.join(routePath)}{marker}")
    return allSimpleRoutes


printBanner("VERIFICATION: ALL SIMPLE ROUTES IN THE AGENT'S GRAPH", "-")
baselineRoutes = listAllSimpleRoutes(baselinePlan)
assert baselineRoutes[0][1] == baselinePlan["cost"], "A cheaper route exists than the one UCS found."
print(f"\nCheapest simple route: {baselineRoutes[0][1]} km | UCS result: {baselinePlan['cost']} km | agree: True")

print("\nSearch tree record (road option -> cumulative cost):")
for previousTown, nextTown, newPathCost, decisionNote in baselinePlan["searchTreeRecord"]:
    print(f"  {previousTown:<11} -> {nextTown:<11} cost = {newPathCost:<4} km | {decisionNote}")


------------------------------------------------------------------------------
VERIFICATION: ALL SIMPLE ROUTES IN THE AGENT'S GRAPH
------------------------------------------------------------------------------
 357 km  Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu  <-- chosen by the agent
 383 km  Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
 419 km  Nairobi -> Narok -> Bomet -> Kisii -> Kisumu
 515 km  Nairobi -> Nyeri -> Nyahururu -> Nakuru -> Mau Summit -> Kericho -> Kisumu
 547 km  Nairobi -> Nakuru -> Mau Summit -> Kericho -> Bomet -> Kisii -> Kisumu
 705 km  Nairobi -> Nyeri -> Nyahururu -> Nakuru -> Mau Summit -> Kericho -> Bomet -> Kisii -> Kisumu

Cheapest simple route: 357 km | UCS result: 357 km | agree: True

Search tree record (road option -> cumulative cost):
  Nairobi     -> Narok       cost = 141  km | new route
  Nairobi     -> Nyeri       cost = 151  km | new route
  Nairobi     -> Nakuru      cost = 161  km | new route
  Narok       -> Bomet       cost = 2

## Step 9: Debugging the Knowledge Base

### 9.1 Known and unknown towns

In [26]:
testTowns = ["Nairobi", "kisumu", "  mau summit ", "Mombasa", "Eldoret", ""]

print("--- Debugging town queries ---")
for town in testTowns:
    shownName = repr(town)
    if isKnownTown(town):
        connections = []
        for neighbourTown, roadDistance in getConnectedTowns(town):
            connections.append(f"{neighbourTown} ({roadDistance} km)")
        print(f"{shownName:<16} -> known as {normalizeName(town)}; connected to {', '.join(connections)}")
    else:
        print(f"{shownName:<16} -> Unknown town: not in the knowledge base.")

--- Debugging town queries ---
'Nairobi'        -> known as Nairobi; connected to Narok (141 km), Nyeri (151 km), Nakuru (161 km)
'kisumu'         -> known as Kisumu; connected to Kericho (84 km), Kisii (114 km)
'  mau summit '  -> known as Mau Summit; connected to Kericho (50 km), Nakuru (62 km)
'Mombasa'        -> Unknown town: not in the knowledge base.
'Eldoret'        -> Unknown town: not in the knowledge base.
''               -> Unknown town: not in the knowledge base.


### 9.2 Unknown roads and invalid updates
Invalid updates must be rejected and leave the KB unchanged.

In [27]:
testRoads = [("Nairobi", "Nakuru"), ("Nakuru", "Nairobi"), ("Nairobi", "Kisumu"), ("Nyeri", "Kisii"), ("Nairobi", "Mombasa")]

print("--- Debugging road queries ---")
for fromTown, toTown in testRoads:
    roadDistance = getRoadDistance(fromTown, toTown)
    distanceText = f"{roadDistance} km" if roadDistance is not None else "no Road fact"
    print(f"{fromTown + ' -> ' + toTown:<20} distance: {distanceText:<13} status: {describeRoadStatus(fromTown, toTown)}")

print("\n--- Debugging invalid updates ---")
roadFactsBefore = dict(knowledgeBase["RoadCondition"])
print("Try: close the (non-existent) Nairobi-Kisumu road")
setRoadCondition("Nairobi", "Kisumu", "Closed")
print("Try: change the destination to Mombasa")
setDestination("Mombasa")

assert knowledgeBase["RoadCondition"] == roadFactsBefore
assert getDestination() == "Kisumu"
print("\nCheck passed: invalid updates were rejected and the KB is unchanged.")

--- Debugging road queries ---
Nairobi -> Nakuru    distance: 161 km        status: Open. Usable: the road is open to the trailer.
Nakuru -> Nairobi    distance: 161 km        status: Open. Usable: the road is open to the trailer.
Nairobi -> Kisumu    distance: no Road fact  status: Unknown road: the KB has no Road fact from Nairobi to Kisumu.
Nyeri -> Kisii       distance: no Road fact  status: Unknown road: the KB has no Road fact from Nyeri to Kisii.
Nairobi -> Mombasa   distance: no Road fact  status: Unknown road: the KB has no Road fact from Nairobi to Mombasa.

--- Debugging invalid updates ---
Try: close the (non-existent) Nairobi-Kisumu road
  Rejected: the KB has no road between Nairobi and Kisumu.
Try: change the destination to Mombasa
  Rejected: 'Mombasa' is not a known town, so Destination(Trailer, Mombasa) was not added.

Check passed: invalid updates were rejected and the KB is unchanged.


### 9.3 Testing the road condition rules
The equivalent of the lecturer's `test_cases` loop.

In [28]:
testConditions = ["Open", "wet", "Under Repair", "Closed", "Flooded", "weight restricted", "Landslide", "Foggy"]

print("--- Debugging road condition rules ---")
for condition in testConditions:
    rule = getConditionRule(condition)
    print(f"Condition: {condition:<18} -> usable: {str(rule['usable']):<5} | {rule['advice']}")

--- Debugging road condition rules ---
Condition: Open               -> usable: True  | Usable: the road is open to the trailer.
Condition: wet                -> usable: True  | Usable: reduce speed, the surface is slippery.
Condition: Under Repair       -> usable: True  | Usable: expect delays from roadworks.
Condition: Closed             -> usable: False | Not usable: the road is closed to all traffic.
Condition: Flooded            -> usable: False | Not usable: the trailer cannot safely cross flood water.
Condition: weight restricted  -> usable: False | Not usable: the loaded trailer exceeds the weight limit.
Condition: Landslide          -> usable: False | Unknown condition: treat the road as unusable until confirmed safe.
Condition: Foggy              -> usable: False | Unknown condition: treat the road as unusable until confirmed safe.


### 9.4 Road closure and replanning
New observation: **Nakuru–Mau Summit is closed.** `observeRoadCondition()` updates the KB. The agent then replans and explains the change.

In [29]:
def observeRoadCondition(firstTown, secondTown, condition):
    """A percept: the trailer observes a road's condition, tells the KB, and reports the rule's conclusion."""
    firstTown = normalizeName(firstTown)
    secondTown = normalizeName(secondTown)
    condition = normalizeName(condition)
    print(f"Observation: the road between {firstTown} and {secondTown} is {condition}.")

    if not setRoadCondition(firstTown, secondTown, condition):
        return False

    print(f"  KB updated: RoadCondition({firstTown}, {secondTown}, {condition}) and RoadCondition({secondTown}, {firstTown}, {condition})")
    print(f"  Rule applied: {getConditionRule(condition)['advice']}")
    print(f"  Inferred: CanUse({firstTown}, {secondTown}) = {canUseRoad(firstTown, secondTown)}")
    return True


def explainRouteChange(previousPlan, newPlan):
    """Explain why the new plan differs from the previous one, using the current KB."""
    printBanner("WHY THE DECISION CHANGED", "-")
    previousRoute = previousPlan["route"]

    blockedSegments = []
    for pathIndex in range(len(previousRoute) - 1):
        fromTown = previousRoute[pathIndex]
        toTown = previousRoute[pathIndex + 1]
        if not canUseRoad(fromTown, toTown):
            condition = knowledgeBase["RoadCondition"][(fromTown, toTown)]
            blockedSegments.append(f"{fromTown} -> {toTown} is now {condition}")

    print("Previous route:", " -> ".join(previousRoute), f"({previousPlan['cost']} km)")
    if len(blockedSegments) == 0:
        print("Every road on the previous route is still usable.")
    else:
        print("The previous route can no longer be used because:", "; ".join(blockedSegments))

    if newPlan["route"] is None:
        print("No usable alternative exists, so the trailer must wait for a road to reopen.")
    else:
        print("New route:     ", " -> ".join(newPlan["route"]), f"({newPlan['cost']} km)")
        print(f"Change in distance: {newPlan['cost'] - previousPlan['cost']:+} km")
        print("Decision: follow the new route. It is the cheapest route left using only roads the KB says are usable.")

**Before:** the 357 km plan.

In [30]:
planBeforeClosure = planRoute()
print("Plan before closure:", " -> ".join(planBeforeClosure["route"]), f"({planBeforeClosure['cost']} km)")

Plan before closure: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km)


**Update the KB and recognise that the road is unavailable.**

In [31]:
observeRoadCondition("Nakuru", "Mau Summit", "Closed")

print()
printAnswer("Is the road from Nakuru to Mau Summit open?", describeRoadStatus("Nakuru", "Mau Summit"), "inference")
printAnswer("Is the road from Mau Summit to Nakuru open?", describeRoadStatus("Mau Summit", "Nakuru"), "inference")
printAnswer("Which roads are unavailable?", listUnavailableRoads(), "inference: RoadClosed rule")
printAnswer("Can the trailer still reach Kisumu?", "Yes" if canReachDestination() else "No", "inference: Reachable rules")

Observation: the road between Nakuru and Mau Summit is Closed.
  KB updated: RoadCondition(Nakuru, Mau Summit, Closed) and RoadCondition(Mau Summit, Nakuru, Closed)
  Rule applied: Not usable: the road is closed to all traffic.
  Inferred: CanUse(Nakuru, Mau Summit) = False

Q: Is the road from Nakuru to Mau Summit open?
A: Closed. Not usable: the road is closed to all traffic.
   [inference]

Q: Is the road from Mau Summit to Nakuru open?
A: Closed. Not usable: the road is closed to all traffic.
   [inference]

Q: Which roads are unavailable?
A: [('Mau Summit', 'Nakuru', 'Closed')]
   [inference: RoadClosed rule]

Q: Can the trailer still reach Kisumu?
A: Yes
   [inference: Reachable rules]



**Updated usable network:** both Nakuru ↔ Mau Summit arcs should be gone.

In [32]:
graphAfterClosure = buildAvailableRoadGraph()

print("Nakuru's usable neighbours:    ", graphAfterClosure["Nakuru"])
print("Mau Summit's usable neighbours:", graphAfterClosure["Mau Summit"])
print(f"Directed arcs in the usable graph: {countArcs(graphAfterClosure)} (was {countArcs(practicalThreeGraph)})")

assert ("Mau Summit", 62) not in graphAfterClosure["Nakuru"]
assert ("Nakuru", 62) not in graphAfterClosure["Mau Summit"]
assert countArcs(graphAfterClosure) == 22

Nakuru's usable neighbours:     [('Nyahururu', 64), ('Nairobi', 161)]
Mau Summit's usable neighbours: [('Kericho', 50)]
Directed arcs in the usable graph: 22 (was 24)


**Replan, find the alternative and explain.**

In [33]:
planAfterClosure = planRoute()
explainPlan(planAfterClosure)
explainRouteChange(planBeforeClosure, planAfterClosure)

assert planAfterClosure["route"] == ["Nairobi", "Narok", "Bomet", "Kericho", "Kisumu"]
assert planAfterClosure["cost"] == 383
print("\nCheck passed: after the closure the agent chose Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (383 km).")


AGENT DECISION: Nairobi -> Kisumu
Route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Narok                               141                 141
Narok -> Bomet                                  81                 222
Bomet -> Kericho                                77                 299
Kericho -> Kisumu                               84                 383

Total distance: 141 + 81 + 77 + 84 = 383 km
Matches the cost returned by UCS: True
Next move: drive from Nairobi to Narok.
Reason: Using only the 11 of 12 roads the KB marks as usable, UCS settled 10 towns and found this to be the lowest-cost route.

------------------------------------------------------------------------------
WHY THE DECISION CHANGED
------------------------------------------------------------------------------
Previous route: Nairobi -> Nakuru -> Mau Summit -

In [34]:
printBanner("VERIFICATION: ALL SIMPLE ROUTES AFTER THE CLOSURE", "-")
routesAfterClosure = listAllSimpleRoutes(planAfterClosure)
assert routesAfterClosure[0][1] == 383


------------------------------------------------------------------------------
VERIFICATION: ALL SIMPLE ROUTES AFTER THE CLOSURE
------------------------------------------------------------------------------
 383 km  Nairobi -> Narok -> Bomet -> Kericho -> Kisumu  <-- chosen by the agent
 419 km  Nairobi -> Narok -> Bomet -> Kisii -> Kisumu


**Contrast:** UCS on the unchanged Practical Exercise 3 graph still uses the closed road.

In [35]:
staticRoute, staticCost, staticRecord, staticSelected, staticExpanded, staticSettled = uniformCostSearch(
    practicalThreeGraph, "Nairobi", "Kisumu", showTrace=False
)
print("UCS on the original Practical Exercise 3 graph (ignores the KB):", " -> ".join(staticRoute), f"({staticCost} km)")
print("Does that route use the closed Nakuru -> Mau Summit road?", "Mau Summit" in staticRoute and "Nakuru" in staticRoute)
print("UCS on the KB's usable graph:                         ", " -> ".join(planAfterClosure["route"]), f"({planAfterClosure['cost']} km)")

assert staticCost == 357
assert planAfterClosure["cost"] != staticCost

UCS on the original Practical Exercise 3 graph (ignores the KB): Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km)
Does that route use the closed Nakuru -> Mau Summit road? True
UCS on the KB's usable graph:                          Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (383 km)


**Reopen the road:** the 357 km route should return.

In [36]:
observeRoadCondition("Nakuru", "Mau Summit", "Open")

planAfterReopening = planRoute()
print()
print("Unavailable roads:", listUnavailableRoads() if len(listUnavailableRoads()) > 0 else "none")
print("Usable graph identical to Practical Exercise 3 again?", buildAvailableRoadGraph() == practicalThreeGraph)
explainPlan(planAfterReopening)

assert isRoadOpen("Nakuru", "Mau Summit") and isRoadOpen("Mau Summit", "Nakuru")
assert planAfterReopening["route"] == baselinePlan["route"]
assert planAfterReopening["cost"] == 357
print("\nCheck passed: reopening the road restored the original 357 km route.")

Observation: the road between Nakuru and Mau Summit is Open.
  KB updated: RoadCondition(Nakuru, Mau Summit, Open) and RoadCondition(Mau Summit, Nakuru, Open)
  Rule applied: Usable: the road is open to the trailer.
  Inferred: CanUse(Nakuru, Mau Summit) = True

Unavailable roads: none
Usable graph identical to Practical Exercise 3 again? True

AGENT DECISION: Nairobi -> Kisumu
Route: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Nakuru                              161                 161
Nakuru -> Mau Summit                            62                 223
Mau Summit -> Kericho                           50                 273
Kericho -> Kisumu                               84                 357

Total distance: 161 + 62 + 50 + 84 = 357 km
Matches the cost returned by UCS: True
Next move: drive from Nairobi to Nakuru.
Reason:

### 9.5 Unreachable destination
Both roads into Kisumu are flooded. The agent should decide not to move.

In [37]:
observeRoadCondition("Kericho", "Kisumu", "Flooded")
observeRoadCondition("Kisii", "Kisumu", "Flooded")

print()
printAnswer("Which towns are reachable from Nairobi?", ", ".join(sorted(findReachableTowns("Nairobi"))), "inference: Reachable rules")
printAnswer("Can the trailer reach Kisumu?", "Yes" if canReachDestination() else "No", "inference")

floodedPlan = planRoute()
explainPlan(floodedPlan)
assert floodedPlan["route"] is None
assert "Kisumu" not in findReachableTowns("Nairobi")

Observation: the road between Kericho and Kisumu is Flooded.
  KB updated: RoadCondition(Kericho, Kisumu, Flooded) and RoadCondition(Kisumu, Kericho, Flooded)
  Rule applied: Not usable: the trailer cannot safely cross flood water.
  Inferred: CanUse(Kericho, Kisumu) = False
Observation: the road between Kisii and Kisumu is Flooded.
  KB updated: RoadCondition(Kisii, Kisumu, Flooded) and RoadCondition(Kisumu, Kisii, Flooded)
  Rule applied: Not usable: the trailer cannot safely cross flood water.
  Inferred: CanUse(Kisii, Kisumu) = False

Q: Which towns are reachable from Nairobi?
A: Bomet, Kericho, Kisii, Mau Summit, Nakuru, Narok, Nyahururu, Nyeri
   [inference: Reachable rules]

Q: Can the trailer reach Kisumu?
A: No
   [inference]


AGENT DECISION: Nairobi -> Kisumu
Decision: do not move. No usable route exists.
Reason: The reachability rules show no chain of usable roads from Nairobi to Kisumu. Unavailable roads: [('Kericho', 'Kisumu', 'Flooded'), ('Kisii', 'Kisumu', 'Flooded')]


The reachability rules detect this before any search runs. Reopen both roads:

In [38]:
observeRoadCondition("Kericho", "Kisumu", "Open")
observeRoadCondition("Kisii", "Kisumu", "Open")

assert canReachDestination()
assert planRoute()["cost"] == 357
print("\nCheck passed: Kisumu is reachable again and the 357 km route is restored.")

Observation: the road between Kericho and Kisumu is Open.
  KB updated: RoadCondition(Kericho, Kisumu, Open) and RoadCondition(Kisumu, Kericho, Open)
  Rule applied: Usable: the road is open to the trailer.
  Inferred: CanUse(Kericho, Kisumu) = True
Observation: the road between Kisii and Kisumu is Open.
  KB updated: RoadCondition(Kisii, Kisumu, Open) and RoadCondition(Kisumu, Kisii, Open)
  Rule applied: Usable: the road is open to the trailer.
  Inferred: CanUse(Kisii, Kisumu) = True

Check passed: Kisumu is reachable again and the 357 km route is restored.


### 9.6 Already at the destination

In [39]:
setTrailerLocation("Kisumu")
explainPlan(planRoute())
assert planRoute()["cost"] == 0

setTrailerLocation("Nairobi")
print("\nTrailer location restored to:", getCurrentLocation())


AGENT DECISION: Kisumu -> Kisumu
Route: Kisumu
Reason: At(Trailer, Kisumu) and Destination(Trailer, Kisumu) agree: no movement is needed.

Trailer location restored to: Nairobi


### 9.7 Debugging summary

In [40]:
finalChecks = [
    ("KB holds 10 towns", len(knowledgeBase["Town"]) == 10),
    ("KB holds 24 directed Road facts", len(knowledgeBase["Road"]) == 24),
    ("All roads usable again", len(listUnavailableRoads()) == 0),
    ("KB graph equals the Practical Exercise 3 graph", buildAvailableRoadGraph() == practicalThreeGraph),
    ("Trailer is At Nairobi", getCurrentLocation() == "Nairobi"),
    ("Destination is Kisumu", getDestination() == "Kisumu"),
    ("Baseline route is 357 km", planRoute()["cost"] == 357),
    ("Closure gave 383 km via Narok and Bomet", planAfterClosure["cost"] == 383),
    ("Flooding made Kisumu unreachable", floodedPlan["route"] is None),
    ("Unknown condition treated as unusable", getConditionRule("Landslide")["usable"] is False),
]

for checkDescription, checkPassed in finalChecks:
    print(f"{'PASS' if checkPassed else 'FAIL'}  {checkDescription}")

allChecksPassed = True
for checkDescription, checkPassed in finalChecks:
    if not checkPassed:
        allChecksPassed = False
assert allChecksPassed

PASS  KB holds 10 towns
PASS  KB holds 24 directed Road facts
PASS  All roads usable again
PASS  KB graph equals the Practical Exercise 3 graph
PASS  Trailer is At Nairobi
PASS  Destination is Kisumu
PASS  Baseline route is 357 km
PASS  Closure gave 383 km via Narok and Bomet
PASS  Flooding made Kisumu unreachable
PASS  Unknown condition treated as unusable


## Optional Extension: Driving the Route and Replanning on the Way
The agent loop is perceive → tell → ask → act. The trailer drives one road at a time and learns of the closure only on reaching Nakuru.

In [41]:
def driveToDestination(observationsOnArrival, maximumMoves=20):
    """
    Drive one road at a time, replanning from the KB before every move.
    observationsOnArrival maps a town to a list of (firstTown, secondTown, condition) percepts
    the trailer makes when it arrives there.
    """
    journey = [getCurrentLocation()]
    distanceTravelled = 0

    for moveNumber in range(1, maximumMoves + 1):
        currentLocation = getCurrentLocation()
        if currentLocation == getDestination():
            print(f"\nArrived at {currentLocation}.")
            break

        # Ask: what does the KB say is the best route from here?
        plan = planRoute()
        if plan["route"] is None:
            print(f"\nStuck at {currentLocation}: {plan['reason']}")
            break

        # Act: drive only the first road of the plan.
        nextTown = plan["route"][1]
        roadDistance = getRoadDistance(currentLocation, nextTown)
        print(f"\nMove {moveNumber}: current plan {' -> '.join(plan['route'])} ({plan['cost']} km)")
        print(f"  Drive {currentLocation} -> {nextTown} ({roadDistance} km)")
        setTrailerLocation(nextTown)
        distanceTravelled = distanceTravelled + roadDistance
        journey.append(nextTown)

        # Perceive and tell: record anything observed on arrival.
        for firstTown, secondTown, condition in observationsOnArrival.get(nextTown, []):
            print("  ", end="")
            observeRoadCondition(firstTown, secondTown, condition)

    return journey, distanceTravelled


observationsOnArrival = {"Nakuru": [("Nakuru", "Mau Summit", "Closed")]}
journey, distanceTravelled = driveToDestination(observationsOnArrival)

print("\nJourney driven:", " -> ".join(journey))
print("Total distance driven:", distanceTravelled, "km")


Move 1: current plan Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km)
  Drive Nairobi -> Nakuru (161 km)
  Observation: the road between Nakuru and Mau Summit is Closed.
  KB updated: RoadCondition(Nakuru, Mau Summit, Closed) and RoadCondition(Mau Summit, Nakuru, Closed)
  Rule applied: Not usable: the road is closed to all traffic.
  Inferred: CanUse(Nakuru, Mau Summit) = False

Move 2: current plan Nakuru -> Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (544 km)
  Drive Nakuru -> Nairobi (161 km)

Move 3: current plan Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (383 km)
  Drive Nairobi -> Narok (141 km)

Move 4: current plan Narok -> Bomet -> Kericho -> Kisumu (242 km)
  Drive Narok -> Bomet (81 km)

Move 5: current plan Bomet -> Kericho -> Kisumu (161 km)
  Drive Bomet -> Kericho (77 km)

Move 6: current plan Kericho -> Kisumu (84 km)
  Drive Kericho -> Kisumu (84 km)

Arrived at Kisumu.

Journey driven: Nairobi -> Nakuru -> Nairobi -> Narok -> Bomet -> Kericho -> 

Learning of the closure late costs 161 + 544 = 705 km instead of 383 km. Now restore the KB:

In [42]:
assert journey == ["Nairobi", "Nakuru", "Nairobi", "Narok", "Bomet", "Kericho", "Kisumu"]
assert distanceTravelled == 161 + 544

setRoadCondition("Nakuru", "Mau Summit", "Open")
setTrailerLocation("Nairobi")
print("KB restored: At(Trailer,", getCurrentLocation() + "); unavailable roads:", listUnavailableRoads() or "none")
assert planRoute()["cost"] == 357

KB restored: At(Trailer, Nairobi); unavailable roads: none


## Step 10: Final Reflection

- **Representation:** a dict keyed by predicate, as in the reference. Two-way roads are stored as two directed facts.
- **Facts vs rules:** facts (conditions, location) change; rules (`roadConditionRules`, `CanUse`, `RoadClosed`, `Reachable`) do not. Derived predicates are computed, never stored, so they cannot contradict the facts.
- **Reasoning:** condition rules decide usability; forward chaining decides reachability; UCS applies the lowest-cost rule.
- **Queries:** answered by fact lookup or by inference. Unknown inputs get explicit answers, and invalid updates are rejected.
- **UCS integration:** Practical Exercise 3's UCS runs on a graph built from the KB. With all roads open this graph equals the Practical Exercise 3 graph (357 km).
- **Response to change:** closure gives 383 km via Narok and Bomet; reopening gives 357 km again; flooding means no move. Unchanged Practical Exercise 3 UCS would have used the closed road.

**Challenges:** deciding what to store and what to infer; reusing Practical Exercise 3's UCS without rewriting it; consistent naming of constants.

**Limitations:** hand-written rules, not a general inference engine; small town-level network; conditions are certain and apply in both directions; distance is the only cost; observations are simulated.

**Improvements:** cost penalties for Wet or Under Repair; trailer weight and height facts; one-way and time-limited closures; uncertain observations; A* for larger networks; live traffic data.